# Pattern 2: Identity Parameter

Building on pattern 1, the MCP server now sends both the shared API key and an X-User-Id header with the username. The API key proves the MCP server is a trusted caller. The X-User-Id asserts who the user is.

The service trusts the identity claim because it trusts the caller.

**What the service sees:** A verified API key (trusted caller) plus a claimed username string.

**Weakness:** The user identity has no cryptographic proof. Any trusted caller can assert any username. If the MCP server is compromised, it can impersonate any user.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p02_identity_param")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭───────────────────────── p02_identity_param/mcp_auth.py ─────────────────────────╮
│    1 """Pattern 2: Identity Parameter.                                           │
│    2                                                                             │
│    3 The MCP server sends the shared API key (proves it's a trusted caller) plus │
│    4 an X-User-Id header with the username. The service trusts the identity      │
│    5 because it trusts the caller.                                               │
│    6 """                                                                         │
│    7                                                                             │
│    8 from framework.config import SHARED_SERVICE_API_KEY                         │
│    9 from framework.mcp.auth import AuthHandler                                  │
│   10                                                                             │
│   11                                                                             │
│   12 class IdentityParamHandler(AuthHandler):                                    │
│   13     async def prepare_request(self, user_context, headers):                 │
│   14         headers["X-API-Key"] = SHARED_SERVICE_API_KEY                       │
│   15         user = user_context.get("user")                                     │
│   16         if user:                                                            │
│   17             headers["X-User-Id"] = user                                     │
│   18         return headers                                                      │
│   19                                                                             │
│   20                                                                             │
│   21 auth_handler = IdentityParamHandler()                                       │
│   22                                                                             │
╰──────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────── p02_identity_param/service_auth.py ──────────────────────────────╮
│    1 """Pattern 2: Identity Parameter (service side).                                         │
│    2                                                                                          │
│    3 The service validates the API key (proves the caller is a trusted MCP                    │
│    4 server), then reads X-User-Id as the asserted user identity. No crypto                   │
│    5 proof, but trusted because the caller is authenticated.                                  │
│    6 """                                                                                      │
│    7                                                                                          │
│    8 from fastapi import Request                                                              │
│    9 from framework.services.identity import Identity                                         │
│   10                                                                                          │
│   11 EXPECTED_API_KEY = "dev-shared-api-key"                                                  │
│   12                                                                                          │
│   13                                                                                          │
│   14 async def get_expense_identity(request: Request) -> Identity:                            │
│   15     api_key = request.headers.get("x-api-key")                                           │
│   16     if not api_key or api_key != EXPECTED_API_KEY:                                       │
│   17         return Identity(method="none", detail="invalid or missing API key")              │
│   18                                                                                          │
│   19     user_id = request.headers.get("x-user-id")                                           │
│   20     if not user_id:                                                                      │
│   21         return Identity(                                                                 │
│   22             method="api_key",                                                            │
│   23             detail="valid API key but no X-User-Id header",                              │
│   24         )                                                                                │
│   25     return Identity(                                                                     │
│   26         method="string_id",                                                              │
│   27         user_id=user_id,                                                                 │
│   28         detail="API key verified caller; X-User-Id accepted on trust (no crypto proof)", │
│   29     )                                                                                    │
│   30                                                                                          │
│   31                                                                                          │
│   32 # Document service uses the same auth                                                    │
│   33 get_document_identity = get_expense_identity                                             │
│   34                                                                                          │
╰───────────────────────────────────────────────────────────────────────────────────────────────╯

The MCP server adds both X-API-Key and X-User-Id to every request.
The service validates the API key, then accepts the X-User-Id on trust.
Compare this to pattern 1 where the service had no user context at all.

In [3]:
await runner.start()

[04/14/26 06:18:41] INFO     StreamableHTTP session manager started                  ]8;id=332974;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=575038;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=188012;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=781160;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:65466/mcp "HTTP/1.1 200 OK"        ]8;id=973845;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=181291;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=731708;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=451652;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=329277;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=86736;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:65466/mcp "HTTP/1.1 200 OK"        ]8;id=918886;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=900222;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:65466/mcp "HTTP/1.1 200 OK"        ]8;id=960089;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=425418;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:65467/mcp "HTTP/1.1 200 OK"        ]8;id=494415;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=663551;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=70968;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=401254;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=343204;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=92274;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my recent expenses?")

                    INFO     HTTP Request: POST                                                     ]8;id=715166;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=679881;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────╮
│ [alice] What are my recent expenses? │
╰──────────────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=994108;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=937700;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=813151;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=250725;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=963352;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=946255;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:18:47] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=907185;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=805583;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:18:51] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=428064;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=504744;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "string_id",                 │
│     │              │      │        │ "identity_detail": "API key verified caller; X-User-Id accepted on trust   │
│     │              │      │        │ (no crypto proof)", "count": 4, "expenses": [{"id": 1, "user_id": "alice", │
│     │              │      │        │ "department": "engineering", "amount": 42.5, "category": "software",       │
│     │              │      │        │ "description": "JetBrain                                                   │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭───────────────────────────────────────── answer ─────────────────────────────────────────╮
│ Here are your recent expenses:                                                           │
│                                                                                          │
│ - ID 1 | $42.50 | software | JetBrains AI assistant subscription | status: approved      │
│ - ID 2 | $156.00 | travel | Train ticket to client offsite | status: approved            │
│ - ID 3 | $89.00 | books | Designing Data-Intensive Applications | status: approved       │
│ - ID 4 | $1,450.00 | hardware | External 4K monitor | status: pending                    │
│                                                                                          │
│ Totals:                                                                                  │
│ - Approved: $287.50                                                                      │
│ - Pending: $1,450.00                                                                     │
│ - Overall: $1,737.50                                                                     │
│                                                                                          │
│ Would you like more details on any item or to take action on the pending expense (ID 4)? │
╰──────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your recent expenses:\n\n- ID 1 | $42.50 | software | JetBrains AI assistant subscription | status: approved\n- ID 2 | $156.00 | travel | Train ticket to client offsite | status: approved\n- ID 3 | $89.00 | books | Designing Data-Intensive Applications | status: approved\n- ID 4 | $1,450.00 | hardware | External 4K monitor | status: pending\n\nTotals:\n- Approved: $287.50\n- Pending: $1,450.00\n- Overall: $1,737.50\n\nWould you like more details on any item or to take action on the pending expense (ID 4)?', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "string_id", "identity_detail": "API key verified caller; X-User-Id accepted on trust (no crypto proof)", "count": 4, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering", "amount": 42.5, "category": "software", "description": "JetBrain', error=None)])

In [5]:
await runner.run_as("bob", "Show me all expenses")

                    INFO     HTTP Request: POST                                                     ]8;id=12047;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=653499;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭────────────────────────────╮
│ [bob] Show me all expenses │
╰────────────────────────────╯

[04/14/26 06:18:52] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=91087;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=932339;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:18:53] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=254704;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=827403;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=629047;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=901665;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

                    INFO     Terminating session: None                                       ]8;id=403405;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=281449;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

[04/14/26 06:18:57] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=256606;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=515604;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "string_id",                 │
│     │              │      │        │ "identity_detail": "API key verified caller; X-User-Id accepted on trust   │
│     │              │      │        │ (no crypto proof)", "count": 2, "expenses": [{"id": 5, "user_id": "bob",   │
│     │              │      │        │ "department": "engineering", "amount": 320.0, "category": "training",      │
│     │              │      │        │ "description": "OAuth 2.0                                                  │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Here are your expenses (2 in total):                                                                            │
│                                                                                                                 │
│ - ID 5 | Department: engineering | Category: training | Amount: $320.00 | Description: OAuth 2.0 deep-dive      │
│ workshop | Status: Approved                                                                                     │
│ - ID 6 | Department: engineering | Category: meals | Amount: $67.00 | Description: Team lunch after the         │
│ migration shipped | Status: Approved                                                                            │
│                                                                                                                 │
│ Total: $387.00                                                                                                  │
│                                                                                                                 │
│ Would you like me to filter by department or export these as CSV?                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your expenses (2 in total):\n\n- ID 5 | Department: engineering | Category: training | Amount: $320.00 | Description: OAuth 2.0 deep-dive workshop | Status: Approved\n- ID 6 | Department: engineering | Category: meals | Amount: $67.00 | Description: Team lunch after the migration shipped | Status: Approved\n\nTotal: $387.00\n\nWould you like me to filter by department or export these as CSV?', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "string_id", "identity_detail": "API key verified caller; X-User-Id accepted on trust (no crypto proof)", "count": 2, "expenses": [{"id": 5, "user_id": "bob", "department": "engineering", "amount": 320.0, "category": "training", "description": "OAuth 2.0', error=None)])

In [6]:
await runner.run_as("dave", "Search for architecture documents")

                    INFO     HTTP Request: POST                                                     ]8;id=905591;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=647564;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────────────────────╮
│ [dave] Search for architecture documents │
╰──────────────────────────────────────────╯

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=231482;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=709842;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=432564;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=814432;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:19:00] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=433465;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=973329;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

...

[04/14/26 06:19:03] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=120401;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=358075;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:19:06] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=23726;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=312403;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool             ┃ args                  ┃ status ┃ result                                                ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ search_documents │ {'q': 'architecture'} │      - │ {'type': 'text', 'text': '{"identity_method":         │
│     │                  │                       │        │ "string_id", "identity_detail": "API key verified     │
│     │                  │                       │        │ caller; X-User-Id accepted on trust (no crypto        │
│     │                  │                       │        │ proof)", "allowed_groups": ["admin", "platform",      │
│     │                  │                       │        │ "public"], "count": 0, "documents": [], "_status":    │
│     │                  │                       │        │ 200}'}                                                │
└─────┴──────────────────┴───────────────────────┴────────┴───────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ I searched for architecture documents, but found no results accessible to you.                                  │
│                                                                                                                 │
│ Would you like me to try a more specific query? Some options:                                                   │
│ - "system architecture"                                                                                         │
│ - "software architecture"                                                                                       │
│ - "enterprise architecture"                                                                                     │
│ - "architecture diagrams"                                                                                       │
│                                                                                                                 │
│ Or tell me a department, project, or document type (e.g., design doc, wiki page) to filter by. I can rerun the  │
│ search with that.                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='I searched for architecture documents, but found no results accessible to you.\n\nWould you like me to try a more specific query? Some options:\n- "system architecture"\n- "software architecture"\n- "enterprise architecture"\n- "architecture diagrams"\n\nOr tell me a department, project, or document type (e.g., design doc, wiki page) to filter by. I can rerun the search with that.', tool_calls=[ToolCallTrace(name='search_documents', args={'q': 'architecture'}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "string_id", "identity_detail": "API key verified caller; X-User-Id accepted on trust (no crypto proof)", "allowed_groups": ["admin", "platform", "public"], "count": 0, "documents": [], "_status": 200}\'}', error=None)])

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [7]:
await runner.show_service_identity()

[04/14/26 06:19:07] INFO     HTTP Request: GET http://127.0.0.1:65464/debug/last-request "HTTP/1.1  ]8;id=719648;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=583043;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭────────────────────── expense-service /debug/last-request ──────────────────────╮
│ method:  string_id                                                              │
│ user_id: bob                                                                    │
│ detail:  API key verified caller; X-User-Id accepted on trust (no crypto proof) │
│                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:65465/debug/last-request "HTTP/1.1  ]8;id=67237;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=893895;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭───────────────────── document-service /debug/last-request ──────────────────────╮
│ method:  string_id                                                              │
│ user_id: dave                                                                   │
│ detail:  API key verified caller; X-User-Id accepted on trust (no crypto proof) │
│                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────╯

Now the service reports method=string_id and filters by user.

### What changed from pattern 1

In pattern 1, the service returned ALL expenses and ALL documents to every user -- it had no idea who was calling. Now alice sees only her own expenses, bob sees only his, and dave sees only his. The service is filtering by the asserted username for the first time.

For documents, the service maps each username to access groups (alice/bob see engineering+public, dave sees platform+admin+public). Dave can now see admin-only documents like compensation bands that were returned to everyone in pattern 1 -- but only because the service has a hardcoded username-to-group mapping, not because of any proven claims.

### The weakness

The identity is unproven. The MCP server asserts it, and the service trusts it because the API key checked out. If the MCP server were compromised, it could claim to be any user.

We need richer user context with claims like role and department.

In [8]:
await runner.stop()

Pattern p02_identity_param stopped.

[04/14/26 06:19:08] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=115795;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=293771;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       